<a href="https://colab.research.google.com/github/samrat-mitra-48/Job-Market-Analysis-with-Python-and-Excel/blob/master/Job_Market_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install python-jobspy
!pip install --upgrade numpy pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 MB 17.5 MB/s eta 0:00:00
  Attempting uninstall: regex
    Found existing installation: regex 2025.11.3
    Uninstalling regex-2025.11.3:
      Successfully uninstalled regex-2025.11.3
  Attempting uninstall: NUMPY
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numpy>=2.0, but

In [ ]:
import pandas as pd
from jobspy import scrape_jobs

try:
    # 1. Scrape data roles including advanced field structures
    raw_results = scrape_jobs(
        site_name=["indeed", "linkedin"],
        search_term="data analyst",
        location="India",
        results_wanted=100000,
        hours_old=168, # 7 days
        country_indeed='India'
    )
    df = pd.DataFrame(raw_results)
    print("Data pulled successfully!")
except Exception as e:
    print(f"Scraper error: {e}. Loading realistic structural sample dataset...")
    # Production fallback data showing exactly how the fields behave
    df = pd.DataFrame([
        {'title': 'Data Analyst', 'company': 'TCS', 'location': 'Bengaluru', 'job_type': 'fulltime', 'is_remote': False, 'description': 'SQL', 'site': 'linkedin'},
        {'title': 'Junior Analyst', 'company': 'Wipro', 'location': 'Remote', 'job_type': 'fulltime', 'is_remote': True, 'description': 'Python', 'site': 'indeed'},
        {'title': 'Intern Analyst', 'company': 'Startup', 'location': 'Mumbai', 'job_type': 'internship', 'is_remote': False, 'description': 'Power BI', 'site': 'linkedin'}
    ])

# 2. Select and clean the target columns
target_cols = ['title', 'company', 'location', 'job_type', 'is_remote', 'site', 'description']
df = df[target_cols].dropna(subset=['title'])

# 3. Standardise Workplace Type (Onsite vs Remote)
def classify_workplace(row):
    if row['is_remote'] == True or str(row['location']).lower() == 'remote':
        return 'Remote'
    elif 'hybrid' in str(row['location']).lower():
        return 'Hybrid'
    else:
        return 'Onsite'

df['workplace_type'] = df.apply(classify_workplace, axis=1)

# 4. Standardise Job Type Formats
df['job_type'] = df['job_type'].fillna('Not Specified')
df['job_type'] = df['job_type'].str.replace('fulltime', 'Full-Time').str.replace('internship', 'Internship')

# 5. Handle Missing Salaries with Proxy Flags
# df['min_amount'] = pd.to_numeric(df['min_amount'], errors='coerce')
# df['max_amount'] = pd.to_numeric(df['max_amount'], errors='coerce')
# df['salary_estimated'] = df['min_amount'].isna().astype(int) # Flags rows where salary wasn't listed

# Fill missing salary values with a 0 marker for easy SQL manipulation later
# df['min_amount'] = df['min_amount'].fillna(0)
# df['max_amount'] = df['max_amount'].fillna(0)

df.head()


2026-08-09 12:40:08,814 - INFO - JobSpy:Linkedin - finished scraping


Data pulled successfully!


,title,company,location,job_type,is_remote,site,description,workplace_type
0,Data Analyst,Skillitize India,"Remote, IN",Full-Time,True,indeed,"Key Responsibilities\n\n* Collect, clean, orga...",Remote
1,Data Analytics Internship,Intilaq Technologies,"KL, IN",Internship,False,indeed,We are inviting passionate freshers to join ou...,Onsite
2,Data Analysis,leventm technologies,"KA, IN",Not Specified,False,indeed,**Job Description:**\n\n\nWe are looking for a...,Onsite
3,"Python Technical Lead - Data Analysis, SQL",HCLTech,"TS, IN",Not Specified,False,indeed,"Hyderabad, Telangana\n \nJob Summary\n \nThe...",Onsite
4,MIS Analyst,NaN,"MH, IN",Full-Time,False,indeed,MIS Analyst\n\nFresher/Experienced\n\nMIS /Adv...,Onsite


In [ ]:
df.shape

(1005, 8)

In [ ]:
import pandas as pd
import re

# --- 1. YOUR EXISTING EXPERIENCE EXTRACTOR ---
def extract_years_of_experience(row):
    text = str(row['description'])
    pattern = r'(\d+)\s*(?:\s*-\s*\d+)?\s*(?:\+|plus)?\s*(?:year|yr)s?\s*(?:of\s*)?(?:experience|exp)?'
    match = re.search(pattern, text, re.IGNORECASE)

    if match:
        extracted_num = int(match.group(1))
        return extracted_num if extracted_num <= 12 else None

    title_lower = str(row['title']).lower()
    if any(word in title_lower for word in ['sr', 'senior', 'lead', 'manager']):
        return 5
    elif any(word in title_lower for word in ['jr', 'junior', 'intern', 'fresher', 'associate']):
        return 0
    else:
        return 2

# Apply experience extraction
df['years_experience'] = df.apply(extract_years_of_experience, axis=1)


# --- 2. GLOBAL CLEANING SHIELD (PREVENTS MISSED MATCHES) ---
# Strips markdown code/symbols to keep text scanning 100% reliable
clean_desc = df['description'].astype(str).str.replace(r'\\|\*|-|_|`', ' ', regex=True)


# --- 3. STREAMLINED 2026 DATA ANALYTICS TOOL MATRIX (NO PREFIXES) ---

# A. Core Programming & Scripting
df['sql'] = clean_desc.str.contains(r'sql|pl\s*sql|t\s*sql|my\s*sql|postgre|ms\s*sql', case=False, na=False, regex=True).astype(int)
df['python'] = clean_desc.str.contains('Python', case=False, na=False).astype(int)
df['r_lang'] = clean_desc.str.contains(r'\bR\b', case=True, na=False).astype(int)

# B. Business Intelligence & Dashboards
df['powerbi'] = clean_desc.str.contains(r'power\s*bi|power-bi|pbi', case=False, na=False, regex=True).astype(int)
df['tableau'] = clean_desc.str.contains('Tableau', case=False, na=False).astype(int)
df['looker'] = clean_desc.str.contains('Looker|Google Data Studio', case=False, na=False).astype(int)

# C. Cloud Data Warehouses
df['aws'] = clean_desc.str.contains('AWS|Amazon Web Services', case=False, na=False).astype(int)
df['azure'] = clean_desc.str.contains('Azure', case=False, na=False).astype(int)
df['snowflake'] = clean_desc.str.contains('Snowflake', case=False, na=False).astype(int)
df['bigquery'] = clean_desc.str.contains(r'bigquery|gcp|google\s*cloud', case=False, na=False, regex=True).astype(int)

# D. Spreadsheets (ONLY ONE RAW NAME COLUMN)
excel_pattern = r'excel|ms\s*excel|microsoft\s*excel|exxcel'
df['excel'] = clean_desc.str.contains(excel_pattern, case=False, na=False, regex=True).astype(int)

# E. Advanced Engineering & AI Systems
# df['dbt'] = clean_desc.str.contains('dbt', case=False, na=False).astype(int)
df['spark'] = clean_desc.str.contains('Spark|Databricks', case=False, na=False).astype(int)
df['git'] = clean_desc.str.contains('Git|GitHub|GitLab', case=False, na=False).astype(int)
df['genai'] = clean_desc.str.contains('GenAI|Generative AI|LLM|OpenAI|Prompt|ChatGPT', case=False, na=False).astype(int)


# --- 4. PRINT RESULTS FOR INSPECTION ---
print("📊 Tool extraction matrix complete (Prefixes removed)!")

# Updated to use direct skill column labels
cols_to_preview = [
    'title', 'years_experience',
    'sql', 'python', 'r_lang',
    'powerbi', 'tableau', 'looker',
    'aws', 'azure', 'snowflake', 'bigquery',
    'excel', 'spark', 'git', 'genai'
]


df[cols_to_preview]


📊 Tool extraction matrix complete (Prefixes removed)!


,title,years_experience,sql,python,r_lang,powerbi,tableau,looker,aws,azure,snowflake,bigquery,excel,spark,git,genai
0,Data Analyst,1.0,1,1,0,1,0,0,0,0,0,0,1,0,0,0
1,Data Analytics Internship,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Data Analysis,2.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,"Python Technical Lead - Data Analysis, SQL",5.0,1,1,0,0,0,0,0,0,0,0,1,0,0,0
4,MIS Analyst,2.0,0,0,0,1,1,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1000,Data Analyst,2.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1001,SAS Developer,2.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1002,Data Analyst,2.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1003,ERP Data Analyst,2.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
pd.set_option('display.max_columns', None)
df.head()

,title,company,location,job_type,is_remote,site,description,workplace_type,years_experience,sql,python,r_lang,powerbi,tableau,looker,aws,azure,snowflake,bigquery,excel,spark,git,genai
0,Data Analyst,Skillitize India,"Remote, IN",Full-Time,True,indeed,"Key Responsibilities\n\n* Collect, clean, orga...",Remote,1.0,1,1,0,1,0,0,0,0,0,0,1,0,0,0
1,Data Analytics Internship,Intilaq Technologies,"KL, IN",Internship,False,indeed,We are inviting passionate freshers to join ou...,Onsite,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Data Analysis,leventm technologies,"KA, IN",Not Specified,False,indeed,**Job Description:**\n\n\nWe are looking for a...,Onsite,2.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,"Python Technical Lead - Data Analysis, SQL",HCLTech,"TS, IN",Not Specified,False,indeed,"Hyderabad, Telangana\n \nJob Summary\n \nThe...",Onsite,5.0,1,1,0,0,0,0,0,0,0,0,1,0,0,0
4,MIS Analyst,NaN,"MH, IN",Full-Time,False,indeed,MIS Analyst\n\nFresher/Experienced\n\nMIS /Adv...,Onsite,2.0,0,0,0,1,1,0,0,0,0,0,1,0,0,0


In [ ]:
df['description'].iloc[2]

"**Job Description:**\n\n\nWe are looking for a skilled HR manager to oversee all aspects of Human Resources practices and processes. You will support business needs and ensure the proper implementation of company strategy and objectives. The goal is to promote corporate values and enable business success through human resources management, including job design, recruitment, performance management, training \\& development, employment cycle changes, talent management, and facilities management services.\n\n\nResponsibilities:\n\n* Enhances the organization's human resources by planning, implementing, and evaluating employee relations and human resources policies, programs, and practices.\n* Maintains the work structure by updating job requirements and job descriptions for all positions.\n* Supports organization staff by establishing and interviewing program; counselling managers on candidate selection; conducting and analyzing exit interviews; and recommending changes.\n* Prepares empl

In [ ]:
import re
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download ALL required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')  # Fixes the LookupError
nltk.download('stopwords')

def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # Remove markdown/escaped characters
    text = re.sub(re.compile(r'\\+'), '', text)
    text = text.replace('**', '').replace('\n', ' ')

    # Remove metadata footers
    text = re.sub(r'job types:.*|pay:.*|work location:.*', '', text)

    # Keep only alphabet letters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenize and remove stopwords
    words = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    cleaned_words = [word for word in words if word not in stop_words]

    return " ".join(cleaned_words)

# Apply to your DataFrame
df['cleaned_description'] = df['description'].apply(clean_text)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:

df['description'].iloc[3]

'Hyderabad, Telangana\n  \nJob Summary\n  \nThe Technical Lead at HCL will be responsible for overseeing and leading the technical aspects of projects related to SQL, data analysis, and python. The role involves providing expertise in these areas to ensure successful project delivery and implementation.\n  \nKey Responsibilities\n  \n1\\. Lead and manage a team of technical professionals working on sql, data analysis, and python projects\n  \n2\\. Design, develop, and implement complex sql queries and scripts for data analysis\n  \n3\\. Analyze data sets using python to extract valuable insights and patterns\n  \n4\\. Collaborate with stakeholders to gather requirements and define project scope\n  \n5\\. Provide technical guidance and support to team members\n  \n6\\. Ensure adherence to best practices in sql, data analysis, and python development\n  \n7\\. Troubleshoot technical issues and provide innovative solutions\n  \nSkill Requirements\n  \n1\\. Proficiency in sql to write compl

In [ ]:

df['cleaned_description'].iloc[3]

'hyderabad telangana job summary technical lead hcl responsible overseeing leading technical aspects projects related sql data analysis python role involves providing expertise areas ensure successful project delivery implementation key responsibilities lead manage team technical professionals working sql data analysis python projects design develop implement complex sql queries scripts data analysis analyze data sets using python extract valuable insights patterns collaborate stakeholders gather requirements define project scope provide technical guidance support team members ensure adherence best practices sql data analysis python development troubleshoot technical issues provide innovative solutions skill requirements proficiency sql write complex queries scripts data manipulation analysis strong data analysis skills interpret derive insights large datasets advanced knowledge python programming data processing analysis visualization excellent problemsolving skills ability think anal

In [ ]:
df.head()

,title,company,location,job_type,is_remote,site,description,workplace_type,years_experience,sql,python,r_lang,powerbi,tableau,looker,aws,azure,snowflake,bigquery,excel,spark,git,genai,cleaned_description
0,Data Analyst,Skillitize India,"Remote, IN",Full-Time,True,indeed,"Key Responsibilities\n\n* Collect, clean, orga...",Remote,1.0,1,1,0,1,0,0,0,0,0,0,1,0,0,0,key responsibilities collect clean organize va...
1,Data Analytics Internship,Intilaq Technologies,"KL, IN",Internship,False,indeed,We are inviting passionate freshers to join ou...,Onsite,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,inviting passionate freshers join data analyti...
2,Data Analysis,leventm technologies,"KA, IN",Not Specified,False,indeed,**Job Description:**\n\n\nWe are looking for a...,Onsite,2.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,job description looking skilled hr manager ove...
3,"Python Technical Lead - Data Analysis, SQL",HCLTech,"TS, IN",Not Specified,False,indeed,"Hyderabad, Telangana\n \nJob Summary\n \nThe...",Onsite,5.0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,hyderabad telangana job summary technical lead...
4,MIS Analyst,NaN,"MH, IN",Full-Time,False,indeed,MIS Analyst\n\nFresher/Experienced\n\nMIS /Adv...,Onsite,2.0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,mis analyst fresherexperienced mis advanced ex...


In [ ]:
# 1. Create an independent backup copy of your main dataframe
df_clean = df.copy()

# 2. Apply your cleaning function to the new dataframe
df_clean['cleaned_description'] = df_clean['description'].apply(clean_text)

# 3. Drop the old column from the new dataframe safely
df_clean.drop(columns=['description'], inplace=True)

# 4. Rename the clean column to 'description' in this new version
df_clean.rename(columns={'cleaned_description': 'description'}, inplace=True)


In [ ]:
df_clean

,title,company,location,job_type,is_remote,site,workplace_type,years_experience,sql,python,r_lang,powerbi,tableau,looker,aws,azure,snowflake,bigquery,excel,spark,git,genai,description
0,Data Analyst,Skillitize India,"Remote, IN",Full-Time,True,indeed,Remote,1.0,1,1,0,1,0,0,0,0,0,0,1,0,0,0,key responsibilities collect clean organize va...
1,Data Analytics Internship,Intilaq Technologies,"KL, IN",Internship,False,indeed,Onsite,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,inviting passionate freshers join data analyti...
2,Data Analysis,leventm technologies,"KA, IN",Not Specified,False,indeed,Onsite,2.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,job description looking skilled hr manager ove...
3,"Python Technical Lead - Data Analysis, SQL",HCLTech,"TS, IN",Not Specified,False,indeed,Onsite,5.0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,hyderabad telangana job summary technical lead...
4,MIS Analyst,NaN,"MH, IN",Full-Time,False,indeed,Onsite,2.0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,mis analyst fresherexperienced mis advanced ex...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1000,Data Analyst,Headout,"Bengaluru, Karnataka, India",Not Specified,False,linkedin,Onsite,2.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
1001,SAS Developer,Infosys,"Hyderabad, Telangana, India",Not Specified,False,linkedin,Onsite,2.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
1002,Data Analyst,iamneo - An NIIT Venture,"Coimbatore, Tamil Nadu, India",Not Specified,False,linkedin,Onsite,2.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
1003,ERP Data Analyst,Greif,,Not Specified,False,linkedin,Onsite,2.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,


In [ ]:
# Save df_clean to a new CSV file
df_clean.to_csv('cleaned_job_data.csv', index=False)


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
pd.read_csv('cleaned_job_data.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_job_data.csv'